# Week 12 Lab: Mixed-Type Integration Practice
Connect familiar preparation steps into one workflow, evaluate its predictions, and test a request you create. Plan for about 45–60 minutes; time will vary with how much you refer to the demo.

Use the demo and DATA_GUIDE.md as references. Complete the TODO code and the brief responses beside the relevant outputs. Setup and branch definitions are provided. Keep the model settings and split unchanged.

Restart the kernel, run all cells in order, and retain outputs before submitting.

## 1. Load and investigate
Run the supplied setup and cleaning cells. Use the displayed missing-value counts and DATA_GUIDE.md to review which fields are available when a request arrives. Optional missing values remain in the data for the pipeline to handle.

In [ ]:
import sys
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
print('Python:', sys.version.split()[0], 'scikit-learn:', sklearn.__version__)
pd.set_option('display.max_columns', None)

In [ ]:
TARGET = 'escalated'
raw_df = pd.read_csv('data/service_requests.csv')
display(raw_df.head())
print(raw_df.shape)

In [ ]:
raw_before = raw_df.copy(deep=True)
working_df = raw_df.copy()
numeric_source = ['requested_units', 'available_units', 'asset_age_years']
invalid = (working_df[numeric_source].lt(0).any(axis=1)
           | ~working_df[TARGET].isin([0, 1]))
rejected_ids = working_df.loc[invalid, 'record_id'].tolist()
working_df = working_df.loc[~invalid].copy()
print('Rejected IDs:', rejected_ids)
print('Retained rows:', len(working_df))
assert len(raw_df) == len(working_df) + len(rejected_ids)
assert working_df['record_id'].is_unique
pd.testing.assert_frame_equal(raw_df, raw_before)
display(working_df.isna().sum().to_frame('missing_values'))
print('Missing or blank summaries:', working_df['summary'].fillna('').str.strip().eq('').sum())

## 2. Separate training and test data
Choose the six predictors documented in DATA_GUIDE.md. Exclude the raw identifier and information available only after the work finishes. The supplied split keeps the test rows separate before any preparation is fitted.

In [ ]:
# TODO: choose only the six predictors listed in DATA_GUIDE.md.
FEATURES = [...]
train_df, test_df = train_test_split(working_df, test_size=0.20, random_state=42, stratify=working_df[TARGET])
X_train, X_test = train_df[FEATURES], test_df[FEATURES]
y_train, y_test = train_df[TARGET], test_df[TARGET]
assert set(train_df['record_id']).isdisjoint(test_df['record_id'])
print('Training rows:', len(X_train), 'Test rows:', len(X_test))
display(pd.DataFrame({'train_share':y_train.value_counts(normalize=True), 'test_share':y_test.value_counts(normalize=True)}))

## 3. Define reusable row preparation
Calculate unit_shortage = max(requested_units - available_units, 0). If either input is missing, keep the result missing until the numeric imputer fills it. The same function runs during training and prediction. Text preparation keeps the input one-dimensional and turns missing/blank text into empty strings.

In [ ]:
def add_shortage(frame):
    # Stateless: each row uses only values supplied in that row.
    result = frame.copy()
    result['unit_shortage'] = ...  # TODO: apply the supplied shortage rule
    return result

def prepare_text(series):
    # ColumnTransformer supplies a one-dimensional Series for the scalar selector.
    return series.fillna('').astype(str).str.strip()

## 4. Connect the feature branches
The branch definitions below are provided. Assemble a `ColumnTransformer` named `preprocess` that routes:

- the four fields in `numeric_columns` to `numeric`;
- `channel` to `nominal`;
- `priority` to `ordinal`;
- `summary` to `text`.

Use branch names `numeric`, `nominal`, `ordinal`, and `text`. Drop other columns and use `sparse_threshold=1.0`. Decide which selector must be a single column name and which must be lists.

Then assemble `model` with steps named `engineer`, `prepare`, and `classifier`. Use `FunctionTransformer(add_shortage, validate=False)` for engineering, your `preprocess` object for preparation, and the supplied classifier. Fit the complete workflow using training data only.

In [ ]:
numeric_columns = ['requested_units', 'available_units', 'asset_age_years', 'unit_shortage']
numeric = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])
nominal = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encode', OneHotEncoder(handle_unknown='ignore'))
])
ordinal = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OrdinalEncoder(categories=[['low', 'normal', 'high']],
                              handle_unknown='use_encoded_value', unknown_value=-1))
])
text = Pipeline([
    ('prepare', FunctionTransformer(prepare_text, validate=False, feature_names_out='one-to-one')),
    ('tfidf', TfidfVectorizer(max_features=100, ngram_range=(1, 1), stop_words=None))
])
classifier = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', random_state=42)

# TODO: assemble the ColumnTransformer using the routing requirements above.
preprocess = ...

# TODO: assemble the outer Pipeline with engineer, prepare, and classifier steps.
model = ...

# TODO: fit the complete model using X_train and y_train.

## 5. Inspect the combined features
Run the supplied inspection cell after fitting your pipeline. It checks row count, feature width, and finite values. Convert only the supplied small slice to dense form.

Report the matrix shape and identify one output feature from each of **two different branches**.

In [ ]:
engineered = model.named_steps['engineer'].transform(X_test)
prepared = model.named_steps['prepare'].transform(engineered)
names = model.named_steps['prepare'].get_feature_names_out()
values = prepared.data if hasattr(prepared, 'toarray') else prepared
assert np.isfinite(values).all()
assert prepared.shape == (len(X_test), len(names))
print('Prepared test shape:', prepared.shape, 'Output type:', type(prepared).__name__)
display(pd.DataFrame({'feature_name':names}).head(16))
small = prepared[:3].toarray() if hasattr(prepared, 'toarray') else prepared[:3]
display(pd.DataFrame(small, columns=names).iloc[:, :10])
print('Only a tiny slice is converted to dense form for inspection.')

**Your feature inspection:**

- Matrix shape:
- One feature and its branch:
- A feature from a different branch:

## 6. Evaluate on held-out rows
The supplied code fits a training-majority baseline and calculates metrics for both workflows on the same test rows. Complete the three TODO comparisons.

Model accuracy must exceed baseline accuracy, and positive-class recall must be at least 0.60. Keep the model settings fixed; do not tune using these test results. A failed criterion is a result to explain.

In [ ]:
pred = model.predict(X_test)
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
base_pred = baseline.predict(X_test)

def metrics(actual, predicted) -> dict[str, float]:
    return {
        'accuracy': float(accuracy_score(actual, predicted)),
        'precision (1)': float(precision_score(actual, predicted, zero_division=0)),
        'recall (1)': float(recall_score(actual, predicted, zero_division=0)),
        'F1 (1)': float(f1_score(actual, predicted, zero_division=0)),
    }

# Keep numeric scores for calculations; use the DataFrame to display the comparison.
model_scores = metrics(y_test, pred)
baseline_scores = metrics(y_test, base_pred)
results = pd.DataFrame({'model': model_scores, 'majority_baseline': baseline_scores})
display(results)

# TODO: compare model accuracy with baseline accuracy (strictly greater).
accuracy_above_baseline = ...
# TODO: check whether positive-class recall meets the 0.60 goal.
recall_meets_target = ...
# TODO: combine both checks so both must pass.
meets_performance_criteria = ...
print('Meets performance criteria (higher accuracy than baseline and recall >= 0.60):', meets_performance_criteria)
cm = confusion_matrix(y_test,pred,labels=[0,1])
display(pd.DataFrame(cm,index=['actual 0','actual 1'],columns=['predicted 0','predicted 1']))
review = test_df[['record_id','summary',TARGET]].copy()
review['predicted'] = pred
mistakes = review.loc[review[TARGET] != review['predicted']]
print('False positives:', cm[0,1], 'False negatives:', cm[1,0])
display(mistakes.head(4))
print('zero_division=0 reports undefined precision/recall as zero. Inspect the confusion matrix too.')

**Your evaluation:** In one short paragraph, report model and baseline accuracy and model recall, state which criteria passed, and use the confusion matrix to explain what the missed escalations mean for planning.

## 7. Test a request of your own
First run the worked example below. It checks a complete request and confirms that prediction preserves the fitted vocabulary and imputation medians.

Then create **one different request** with all six predictor columns. Include at least one of these conditions: an optional missing numeric value, missing or blank text, text with no recognized vocabulary, or a channel absent from training. Use a documented priority value.

Before running your case, explain what you expect for its `unit_shortage` and the branch affected by your chosen condition. For missing numbers use `np.nan`. If either shortage input is missing, the engineered shortage should remain missing until imputation.

Complete the TODO cell to engineer and transform your request with the fitted workflow, check the expected shortage, check unchanged feature width and finite values, and display its prediction. Reuse the fitted objects; do not call `fit` or `fit_transform`.

In [ ]:
example_case = pd.DataFrame([{
    'requested_units': 8., 'available_units': 3., 'asset_age_years': 9.,
    'priority': 'high', 'channel': 'portal',
    'summary': 'Intermittent failure during operation'
}])
fitted = model.named_steps['prepare']
vocab_before = fitted.named_transformers_['text'].named_steps['tfidf'].get_feature_names_out().copy()
medians_before = fitted.named_transformers_['numeric'].named_steps['impute'].statistics_.copy()
example_engineered = model.named_steps['engineer'].transform(example_case)
assert example_engineered['unit_shortage'].iloc[0] == 5
example_prepared = fitted.transform(example_engineered)
assert example_prepared.shape[1] == prepared.shape[1]
example_values = example_prepared.data if hasattr(example_prepared, 'toarray') else example_prepared
assert np.isfinite(example_values).all()
example_results = example_case.copy()
example_results['predicted'] = np.asarray(model.predict(example_case))
display(example_results)
assert np.array_equal(vocab_before, fitted.named_transformers_['text'].named_steps['tfidf'].get_feature_names_out())
assert np.array_equal(medians_before, fitted.named_transformers_['numeric'].named_steps['impute'].statistics_)
print('PASS: expected shortage, stable feature width, finite values, and unchanged fitted state.')

**Your test plan (complete before running your case):**

- Condition being tested:
- Expected engineered shortage:
- Expected behavior of the affected branch:

In [ ]:
# TODO: create one different request with all six predictor columns.
my_case = pd.DataFrame([{ }])

# TODO: apply the fitted engineering and preparation steps.
my_engineered = ...
my_prepared = ...

# TODO: assert the expected shortage (use pd.isna when missing is expected).

# TODO: assert unchanged feature width and finite transformed values.

# TODO: predict using model, then copy my_case, add a predicted column,
# and display the results. See the worked example for the display pattern.

# Confirm that your prediction did not change learned preparation.
assert np.array_equal(vocab_before, fitted.named_transformers_['text'].named_steps['tfidf'].get_feature_names_out())
assert np.array_equal(medians_before, fitted.named_transformers_['numeric'].named_steps['impute'].statistics_)

**Your test findings:** What happened? Use your output to compare the result with your expectation. If the text vector is all zero, explain which other feature branches can still contribute.

## 8. Reflect
What would you check first if this workflow failed on new data?

**Your response:**